# 12 - CLIP ViT-L/14 モデル評価

## 概要
OpenAIのCLIP ViT-L/14モデルを評価する。

## モデル情報
- **Model**: openai/clip-vit-large-patch14
- **Embedding dimension**: 768
- **Type**: Multimodal (Image + Text)
- **特徴**: 400M以上の画像-テキストペアで訓練された標準的なCLIPモデル

In [4]:
import time
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.notebook import tqdm

from image_vector_poc import CLIPEmbedder
from image_vector_poc.evaluation import EvaluationReporter, evaluate_embeddings

## 設定

In [5]:
DB_PATH = Path("../data/images.duckdb")
OUTPUT_DIR = Path("../data/evaluations")
BATCH_SIZE = 32
RANDOM_STATE = 42

## 画像カタログの読み込み

In [6]:
conn = duckdb.connect(str(DB_PATH), read_only=True)
query = """
    SELECT id, file_path, category, file_name
    FROM image_catalog
    ORDER BY category, file_name
"""
catalog = conn.execute(query).fetchall()
conn.close()

image_ids = [r[0] for r in catalog]
file_paths = [r[1] for r in catalog]
categories = [r[2] for r in catalog]

category_labels = np.array(categories)
category_counts = pd.Series(categories).value_counts().to_dict()
unique_categories = list(category_counts.keys())

print(f"Total images: {len(catalog)}")
print(f"Categories: {unique_categories}")

Total images: 385
Categories: ['EuroPython2025', 'PyConJP2025', 'PyConJP2025-PreCampHiroshima', 'KashiwaVillagePark2026', 'TokyoNight202505', 'terada']


## モデルの初期化

In [7]:
print("Loading CLIP ViT-L/14 model...")
embedder = CLIPEmbedder(device="cuda")
print(f"Model: {embedder.model_name}")
print(f"Embedding dimension: {embedder.embedding_dim}")
print(f"Device: {embedder.device}")

Loading CLIP ViT-L/14 model...


Loading weights:   0%|          | 0/590 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-large-patch14
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


Model: openai/clip-vit-large-patch14
Embedding dimension: 768
Device: cuda


## 画像のベクトル化

In [8]:
print(f"Generating embeddings for {len(file_paths)} images...")

start_time = time.time()
embeddings_list = []

for i in tqdm(range(0, len(file_paths), BATCH_SIZE), desc="Embedding"):
    batch_paths = file_paths[i:i + BATCH_SIZE]
    batch_images = []
    for path in batch_paths:
        try:
            img = Image.open(path).convert("RGB")
            batch_images.append(img)
        except Exception as e:
            print(f"Error loading {path}: {e}")
            batch_images.append(Image.new("RGB", (224, 224), color="gray"))
    
    batch_embs = embedder.embed_images(batch_images)
    embeddings_list.append(batch_embs)

embeddings = np.vstack(embeddings_list)
processing_time = time.time() - start_time

print(f"\nEmbeddings shape: {embeddings.shape}")
print(f"Processing time: {processing_time:.2f}s")
print(f"Speed: {len(file_paths) / processing_time:.1f} images/sec")

Generating embeddings for 385 images...


Embedding:   0%|          | 0/13 [00:00<?, ?it/s]


Embeddings shape: (385, 768)
Processing time: 87.90s
Speed: 4.4 images/sec


## 評価の実行

In [9]:
print("Running evaluation...\n")

metrics = evaluate_embeddings(
    embeddings=embeddings,
    labels=category_labels,
    model_name=embedder.model_name,
    embedding_dim=embedder.embedding_dim,
    categories=unique_categories,
    category_counts=category_counts,
    processing_time=processing_time,
    random_state=RANDOM_STATE,
)

Running evaluation...



## 結果の表示

In [10]:
print("=" * 60)
print("Evaluation Results - CLIP ViT-L/14")
print("=" * 60)
print(f"Model: {metrics.model_name}")
print(f"Embedding dimension: {metrics.embedding_dim}")
print(f"Processing time: {metrics.processing_time_seconds:.2f}s")

print("\n--- t-SNE Metrics ---")
print(f"2D: Silhouette={metrics.silhouette_2d:.4f}, Trust={metrics.trustworthiness_2d:.4f}, DistRatio={metrics.distance_ratio_2d:.4f}")
print(f"3D: Silhouette={metrics.silhouette_3d:.4f}, Trust={metrics.trustworthiness_3d:.4f}, DistRatio={metrics.distance_ratio_3d:.4f}")

print("\n--- PCA Metrics ---")
print(f"2D: Silhouette={metrics.pca_silhouette_2d:.4f}, Variance={metrics.pca_variance_ratio_2d:.4f}")
print(f"3D: Silhouette={metrics.pca_silhouette_3d:.4f}, Variance={metrics.pca_variance_ratio_3d:.4f}")

Evaluation Results - CLIP ViT-L/14
Model: openai/clip-vit-large-patch14
Embedding dimension: 768
Processing time: 87.90s

--- t-SNE Metrics ---
2D: Silhouette=0.2669, Trust=0.9662, DistRatio=0.3923
3D: Silhouette=0.2106, Trust=0.9740, DistRatio=0.5232

--- PCA Metrics ---
2D: Silhouette=0.1472, Variance=0.2563
3D: Silhouette=0.1832, Variance=0.3055


## 結果の保存

In [11]:
reporter = EvaluationReporter(OUTPUT_DIR)
filepath = reporter.save(metrics)
print(f"Results saved to: {filepath}")

Results saved to: ../data/evaluations/openai_clip-vit-large-patch14_2026-02-01.json


## GPUメモリのクリーンアップ

In [12]:
del embedder
del embeddings

import torch
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("GPU memory cleared.")

GPU memory cleared.


## CLIP ViT-L/14 評価結果の分析

### SigLIP（ベースライン）との比較

| 指標 | SigLIP | CLIP | 差分 | 評価 |
|------|--------|------|------|------|
| シルエット (2D) | 0.132 | **0.267** | +0.135 | CLIP優位 |
| Trustworthiness (2D) | 0.954 | **0.966** | +0.012 | CLIP優位 |
| 距離比 (2D) | 0.487 | **0.392** | -0.095 | CLIP優位 |
| PCA寄与率 (2D) | 24.2% | **25.6%** | +1.4% | ほぼ同等 |
| 処理速度 | 4.5 img/s | 4.4 img/s | -0.1 | ほぼ同等 |

### 詳細分析

**1. クラスタリング品質（シルエットスコア: 0.267）**
- SigLIP (0.132) と比較して **約2倍** のスコア
- カテゴリ間の分離がより明確
- CLIPの大規模訓練データ（400M+ペア）の効果が現れている

**2. 構造保持度（Trustworthiness: 0.966）**
- SigLIP (0.954) より高い値
- 高次元空間の近傍関係をより忠実に保持
- 0.96以上は非常に優秀な水準

**3. カテゴリ分離度（距離比: 0.392）**
- SigLIP (0.487) より **約20%改善**
- 同カテゴリ内の画像がより密集
- イベント・風景の区別がより明確

**4. 処理性能**
- 処理速度はSigLIPとほぼ同等
- 同じ768次元出力で比較しやすい

### 結論

CLIP ViT-L/14 は SigLIP に対して**全ての評価指標で優位**を示した。
特にシルエットスコアの改善（+0.135）は顕著であり、カテゴリ分類タスクにおいてより高い精度が期待できる。